# Week 5 Lab 04: Instrument the Cordwell Pipeline
## Experiment tracking with Weights & Biases

This week produced three working artifacts: a LoRA adapter (Module 01), a
vector index (Module 02), and a RAG pipeline with an evaluation harness
(Module 03). What it did not produce is a record. If someone asked which
configuration produced which faithfulness number, the honest answer today
is a notebook cell somewhere. This lab fixes that.

You will wrap the week's evaluation in Weights and Biases tracking, version
the things the pipeline produces, run a small controlled experiment, and
turn the comparison into a written recommendation backed by run data. The
pipeline itself is pre-written in `lab_support.py` and compressed to run in
seconds. It is not the lesson. The lesson is the record.

**Objectives.** By the end of this lab you can:

1. Initialize a tracked run with a config that passes the re-run test:
   someone could reproduce the result from the config alone.
2. Log metrics with stage prefixes and set a headline summary value.
3. Version an adapter and an eval set as artifacts with metadata, and
   record artifact consumption for lineage.
4. Run a controlled three-way experiment and attribute each metric change
   to the one configuration change that caused it.
5. Build a per-query table, find the worst failures, and (stretch) run a
   small grid sweep and defend a pick from its frontier.

**Time budget (about 3.5 hours core, 30 min stretch).**

| Part | Focus | Time |
|---|---|---|
| A | Backend setup and a hello world run | 20 min |
| B | Instrument the RAG eval: config and metrics | 45 min |
| C | Artifacts: adapter and eval set | 40 min |
| D | Three controlled runs | 45 min |
| E | Comparison and recommendation | 30 min |
| F | Per-query table and worst failures | 30 min |
| G | Stretch: manual grid sweep | 30 min |

Run every cell in order. Checks print PASS, FAIL, or TODO and never crash
the notebook. There are 32 checks; a fresh notebook starts at 3.


### Backends

The lab runs against one of three W&B backends, selected by environment
variable **before you start Jupyter**. Offline is the default and is what
the checks are calibrated against. Every line of student code is identical
across all three; only environment variables change. That is the point the
slides made about developing against a local instance.

| `WANDB_LAB_BACKEND` | What happens | Needs |
|---|---|---|
| `offline` (default) | `WANDB_MODE=offline`; runs written to `./wandb/offline-run-*`, syncable later | Nothing |
| `local` | `WANDB_BASE_URL=http://localhost:8080`; full UI, data stays on your machine | Docker W&B server, see `setup/LOCAL_SERVER_SETUP.md` |
| `cloud` | Standard hosted endpoint | Free account, `wandb login` |

Two things do not work without a server and the lab handles both honestly:
`run.use_artifact` (Part C records lineage in config instead) and
`wandb.sweep` (Part G runs a manual grid instead, and shows the sweep code
for the local backend).


In [ ]:
%pip install -r requirements.txt

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

import lab_support as ls

# Reads WANDB_LAB_BACKEND and sets the wandb environment variables.
# Must run before wandb is imported: wandb reads them at import and
# init time. Same discipline as the MPS fallback variable in Module 01.
BACKEND = ls.configure_wandb_backend()
print(f"Backend: {BACKEND}")

import wandb

print(f"wandb version: {wandb.__version__}")
print(f"Corpus: {len(ls.CORPUS)} documents, eval set: {len(ls.EVAL_QUERIES)} queries")

In [ ]:
# Constants for this lab. Model IDs live in config variables, never
# hard-coded at call sites.
PROJECT = "cordwell-rag"
BASE_MODEL = str(Path("~/models/SmolLM2-360M-Instruct").expanduser().resolve())
ADAPTER_REF = "cordwell-adapter:v0"
JUDGE_MODEL = "deterministic-judge-v1 (offline stand-in)"
TEMPERATURE = 0.1
MAX_NEW_TOKENS = 256
CHUNK_OVERLAP = 48

# The eval set has a semantic version (v2, second revision of the query
# set) that is separate from whatever artifact version counter W&B
# assigns when we log it in Part C. Both belong in the record.
print(f"Eval set semantic version: {ls.EVAL_SET_VERSION}")

In [ ]:
# Check harness. Same contract as Labs 01 through 03: checks print
# PASS, FAIL, or TODO and never crash the notebook. A check that
# raises NameError or NotImplementedError reports TODO, because the
# code it needs has not been written yet.

CHECK_RESULTS = {}


def check(name, fn, detail=""):
    try:
        ok = bool(fn())
    except (NameError, NotImplementedError):
        CHECK_RESULTS[name] = "TODO"
        print(f"TODO {name}")
        return
    except Exception as exc:
        CHECK_RESULTS[name] = "FAIL"
        print(f"FAIL {name} ({type(exc).__name__}: {exc})")
        return
    status = "PASS" if ok else "FAIL"
    CHECK_RESULTS[name] = status
    line = f"{status} {name}"
    if detail and not ok:
        line += f" | expected: {detail}"
    print(line)


def req(value):
    """Guard for check lambdas: a None sentinel means the code that
    should have produced this value is still a TODO."""
    if value is None:
        raise NotImplementedError
    return value


def report():
    from collections import Counter

    counts = Counter(CHECK_RESULTS.values())
    total = len(CHECK_RESULTS)
    print(
        f"{counts.get('PASS', 0)} PASS, {counts.get('FAIL', 0)} FAIL, "
        f"{counts.get('TODO', 0)} TODO of {total} checks run"
    )


check("setup: backend is valid", lambda: BACKEND in ls.VALID_BACKENDS)
check(
    "setup: wandb version is current enough",
    lambda: tuple(int(p) for p in wandb.__version__.split(".")[:2]) >= (0, 21),
    "wandb 0.21 or newer",
)
check(
    "setup: corpus and eval set loaded",
    lambda: len(ls.CORPUS) == 24 and len(ls.EVAL_QUERIES) == 12,
)

## Part A: Backend check and a hello world run

Fail fast on setup. Before instrumenting anything real, confirm the backend
works with the smallest possible run.

**Worked target output.** When `do_hello_run` is correct, the next cell
prints something with this exact shape (the id varies, everything else is
literal):

```text
hello run id: 1a2b3c4d
config: {'purpose': 'setup check'}
summary setup/ok: 1.0
```

On the `local` backend you will also see the run appear in the project
`cordwell-rag` at http://localhost:8080 within a few seconds. On `offline`
the run lands in `./wandb/offline-run-*` instead, and `wandb sync` can push
it to a server later.


In [ ]:
# Given: on the local backend, confirm something is listening before
# any run is attempted. A connection failure here should stop you now,
# not twenty minutes from now.
if BACKEND == "local":
    if ls.server_reachable():
        print("Local W&B server is reachable on port 8080.")
    else:
        raise RuntimeError(
            "WANDB_LAB_BACKEND=local but nothing is listening on "
            "localhost:8080. Start the Docker server first; see "
            "setup/LOCAL_SERVER_SETUP.md."
        )
else:
    print(f"Backend is {BACKEND}; no local server needed.")

In [ ]:
def do_hello_run() -> dict:
    """Create, log to, and finish the smallest possible tracked run.

    Steps:
    1. Start a run with wandb.init: project=PROJECT, name="hello-world",
       config={"purpose": "setup check"}, tags=["setup"].
    2. Log one metric: {"setup/ok": 1.0}.
    3. Snapshot what the run recorded BEFORE finishing it, because the
       live run object is the only view that works on every backend:
       {"id": run.id, "config": dict(run.config),
        "summary": dict(run.summary)}
    4. Finish the run.
    5. Return the snapshot dict.
    """
    raise NotImplementedError("Part A: implement do_hello_run")

In [ ]:
try:
    hello = do_hello_run()
    print(f"hello run id: {hello['id']}")
    print(f"config: {hello['config']}")
    print(f"summary setup/ok: {hello['summary'].get('setup/ok')}")
except NotImplementedError:
    hello = None
    print("TODO: implement do_hello_run above, then re-run this cell")

check("A1: hello run has an id", lambda: isinstance(req(hello)["id"], str) and len(hello["id"]) >= 8)
check("A2: hello config recorded", lambda: req(hello)["config"].get("purpose") == "setup check")
check("A3: hello metric in summary", lambda: float(req(hello)["summary"].get("setup/ok")) == 1.0)

## Part B: Instrument the RAG evaluation

Objective 2. The evaluation itself is Module 03's, pre-written and
compressed. You add the record: a config that passes the re-run test, the
five metrics with stage prefixes, and a headline summary value.

**Worked target output.** `ls.run_rag_eval` returns aggregate metrics and
per-query rows. Here is the real aggregate for the adapter on, `top_k=5`,
`chunk_size=384`, run in the next cell so you see it before writing any
tracking code:

```python
{'context_recall': 1.0, 'context_precision': 0.24,
 'faithfulness': 1.0, 'answer_relevancy': 0.8346,
 'abstention_rate': 0.1667, 'correct_abstention': 1.0}
```

And here is the shape of a complete config, from the slides. Every key is
something that would change the result. The test: could someone re-run
from this dict alone?

```python
{"base_model": ..., "adapter": ..., "adapter_active": ...,
 "temperature": ..., "max_new_tokens": ...,
 "embedding_model": ..., "chunk_size": ..., "chunk_overlap": ..., "top_k": ...,
 "eval_set": ..., "judge_model": ...}
```

Metric names are a contract too. The UI groups by prefix, and the prefix
forces you to declare which stage a metric belongs to:

```text
retrieval/context_recall     retrieval/context_precision
generation/faithfulness      generation/answer_relevancy
generation/abstention_rate
```


In [ ]:
# Given: run the evaluation once, untracked, so you can see what you
# are about to record. This is the entire Module 03 pipeline: chunk,
# embed, retrieve, generate, judge. Deterministic, so your numbers
# match this notebook exactly.
baseline_results = ls.run_rag_eval(adapter_active=True, top_k=5, chunk_size=384)
print(json.dumps(baseline_results["aggregate"], indent=2))
print(f"Per-query rows: {len(baseline_results['per_query'])}")

In [ ]:
def make_config(adapter_active: bool, top_k: int, chunk_size: int) -> dict:
    """Build the complete run config for one Cordwell RAG evaluation.

    Returns a dict with exactly these 11 keys, grouped by stage:

    generation: "base_model" (BASE_MODEL), "adapter" (ADAPTER_REF),
        "adapter_active" (argument), "temperature" (TEMPERATURE),
        "max_new_tokens" (MAX_NEW_TOKENS)
    retrieval: "embedding_model" (ls.EMBEDDING_MODEL_NAME),
        "chunk_size" (argument), "chunk_overlap" (CHUNK_OVERLAP),
        "top_k" (argument)
    evaluation: "eval_set" (ls.EVAL_SET_VERSION),
        "judge_model" (JUDGE_MODEL)

    The three arguments are the three knobs this lab turns. Everything
    else is constant, and it goes in the config anyway: constant today
    is not constant next month, and the config is what proves it.
    """
    raise NotImplementedError("Part B: implement make_config")

In [ ]:
def log_eval_metrics(run, aggregate: dict) -> None:
    """Log the five aggregate metrics to a live run, with stage prefixes.

    One run.log call with this exact mapping:
        retrieval/context_recall      from aggregate["context_recall"]
        retrieval/context_precision   from aggregate["context_precision"]
        generation/faithfulness       from aggregate["faithfulness"]
        generation/answer_relevancy   from aggregate["answer_relevancy"]
        generation/abstention_rate    from aggregate["abstention_rate"]

    Then set the headline summary value the comparison table sorts on:
        run.summary["headline/faithfulness"] = aggregate["faithfulness"]

    These are the Module 03 metrics, not BLEU and ROUGE. Surface
    overlap cannot measure grounding; log the metrics that measure the
    thing you care about.
    """
    raise NotImplementedError("Part B: implement log_eval_metrics")

In [ ]:
try:
    _run = wandb.init(
        project=PROJECT,
        name="rag-eval-dev",
        config=make_config(adapter_active=True, top_k=5, chunk_size=384),
        tags=["rag-eval", "module-04"],
    )
    log_eval_metrics(_run, baseline_results["aggregate"])
    dev_snapshot = {"config": dict(_run.config), "summary": dict(_run.summary)}
    _run.finish()
    print("Config keys:", sorted(dev_snapshot["config"]))
    print("Logged metrics:", sorted(k for k in dev_snapshot["summary"] if "/" in k))
except NotImplementedError as exc:
    dev_snapshot = None
    if "_run" in dir() and wandb.run is not None:
        wandb.run.finish()
    print(f"TODO: {exc}")

REQUIRED_CONFIG_KEYS = {
    "base_model", "adapter", "adapter_active", "temperature",
    "max_new_tokens", "embedding_model", "chunk_size", "chunk_overlap",
    "top_k", "eval_set", "judge_model",
}
METRIC_KEYS = {
    "retrieval/context_recall", "retrieval/context_precision",
    "generation/faithfulness", "generation/answer_relevancy",
    "generation/abstention_rate",
}

check(
    "B1: config has all 11 required keys",
    lambda: set(make_config(True, 5, 384)) == REQUIRED_CONFIG_KEYS,
    "exactly the 11 keys in the docstring",
)
check(
    "B2: config arguments pass through",
    lambda: (
        make_config(False, 3, 256)["adapter_active"] is False
        and make_config(False, 3, 256)["top_k"] == 3
        and make_config(False, 3, 256)["chunk_size"] == 256
    ),
)
check(
    "B3: config records eval set and judge",
    lambda: (
        make_config(True, 5, 384)["eval_set"] == ls.EVAL_SET_VERSION
        and make_config(True, 5, 384)["judge_model"] == JUDGE_MODEL
    ),
)
check(
    "B4: run config was recorded",
    lambda: REQUIRED_CONFIG_KEYS <= set(req(dev_snapshot)["config"]),
)
check(
    "B5: all five metrics logged with exact prefixed names",
    lambda: METRIC_KEYS <= set(req(dev_snapshot)["summary"]),
    "see the docstring metric mapping",
)
check(
    "B6: headline summary set and faithfulness is 1.0",
    lambda: float(req(dev_snapshot)["summary"]["headline/faithfulness"]) == 1.0,
)

## Part C: Version the adapter and the eval set as artifacts

Objective 3. A metric says a number happened. An artifact is the thing
that produced it, and the connection between the two is what lineage
provides. You will log the Module 01 adapter (a faithful miniature is
staged for you) as a `model` artifact and the eval query set as a
`dataset` artifact, then record consumption.

**Worked target output.** A correctly built adapter artifact prints like
this before logging:

```text
artifact: cordwell-adapter | type: model
metadata: {'lora_r': 16, 'lora_alpha': 32, 'base_model': 'HuggingFaceTB/SmolLM2-360M-Instruct'}
files staged: 2
```

**One honest limitation.** `run.use_artifact` needs a server to resolve a
name like `cordwell-adapter:v0`, so it raises `TypeError` on the offline
backend. This is a real property of the tool, not a lab shortcut. The
consumption function below therefore branches: on `local` and `cloud` it
uses the real lineage API; on `offline` it records the input references in
config, which preserves the information and syncs later.


In [ ]:
# Given: stage the files. The adapter directory mirrors what
# SFTTrainer wrote in Module 01 (a config json and a weights file);
# the eval set is the 12 queries as JSONL.
adapter_dir = ls.write_adapter_dir()
eval_file = ls.write_eval_set_file()
print(f"Adapter staged at: {adapter_dir}")
print(f"Eval set staged at: {eval_file}")
for p in sorted(Path(adapter_dir).iterdir()):
    print(f"  {p.name}: {p.stat().st_size} bytes")

In [ ]:
def build_adapter_artifact(adapter_path: str) -> wandb.Artifact:
    """Build (but do not log) the adapter artifact.

    Steps:
    1. Create a wandb.Artifact with name="cordwell-adapter",
       type="model", and metadata
       {"lora_r": 16, "lora_alpha": 32, "base_model": BASE_MODEL}.
       The metadata is what lets a comparison table answer "which rank
       was this adapter?" without downloading anything.
    2. art.add_dir(adapter_path) to stage every file in the directory.
    3. Return the artifact.
    """
    raise NotImplementedError("Part C: implement build_adapter_artifact")


def build_eval_set_artifact(eval_path: str) -> wandb.Artifact:
    """Build (but do not log) the eval set artifact.

    Steps:
    1. Create a wandb.Artifact with name="cordwell-eval",
       type="dataset", and metadata
       {"semantic_version": ls.EVAL_SET_VERSION,
        "num_queries": len(ls.EVAL_QUERIES)}.
    2. art.add_file(eval_path) to stage the single JSONL file.
    3. Return the artifact.
    """
    raise NotImplementedError("Part C: implement build_eval_set_artifact")

In [ ]:
def record_input_artifacts(run) -> str:
    """Record that this run consumes the adapter and eval set.

    On backends with a server ("local", "cloud"):
        run.use_artifact("cordwell-adapter:v0")
        run.use_artifact("cordwell-eval:v0")
        return "use_artifact"
    On "offline" (use_artifact raises TypeError without a server):
        run.config.update({
            "input_adapter_artifact": "cordwell-adapter:v0",
            "input_eval_artifact": "cordwell-eval:v0",
        })
        return "config-lineage"

    Branch on the global BACKEND. Both paths preserve the same
    information; only the first paints the lineage graph in the UI.
    """
    raise NotImplementedError("Part C: implement record_input_artifacts")

In [ ]:
try:
    adapter_art = build_adapter_artifact(adapter_dir)
    eval_art = build_eval_set_artifact(eval_file)
    print(f"artifact: {adapter_art.name} | type: {adapter_art.type}")
    print(f"metadata: {dict(adapter_art.metadata)}")
    print(f"files staged: {len(adapter_art.manifest.entries)}")

    _run = wandb.init(project=PROJECT, name="artifact-registration", tags=["artifacts"])
    logged_adapter = _run.log_artifact(adapter_art)
    logged_eval = _run.log_artifact(eval_art)
    # Wait for artifacts to be finalized before finishing
    logged_adapter.wait()
    logged_eval.wait()
    lineage_mode = record_input_artifacts(_run)
    lineage_config = dict(_run.config)
    _run.finish()
    print(f"Both artifacts logged. Lineage mode: {lineage_mode}")
except NotImplementedError as exc:
    adapter_art = eval_art = None
    lineage_mode = None
    lineage_config = {}
    if wandb.run is not None:
        wandb.run.finish()
    print(f"TODO: {exc}")

check(
    "C1: adapter artifact is a model with LoRA metadata",
    lambda: req(adapter_art).type == "model"
    and adapter_art.metadata["lora_r"] == 16
    and adapter_art.metadata["base_model"] == BASE_MODEL,
)
check(
    "C2: adapter artifact stages the directory contents",
    lambda: len(req(adapter_art).manifest.entries) >= 2,
)
check(
    "C3: eval set artifact is a dataset with a query count",
    lambda: req(eval_art).type == "dataset" and eval_art.metadata["num_queries"] == 12,
)
check(
    "C4: eval artifact records the semantic version",
    lambda: req(eval_art).metadata["semantic_version"] == ls.EVAL_SET_VERSION,
)
check(
    "C5: lineage recorded appropriately for this backend",
    lambda: (
        req(lineage_mode) == "config-lineage"
        and lineage_config.get("input_adapter_artifact") == "cordwell-adapter:v0"
    )
    if BACKEND == "offline"
    else req(lineage_mode) == "use_artifact",
)

## Part D: Three controlled runs

The controlled experiment. Three evaluations, each differing from the
baseline pair by exactly one variable, so that every metric difference in
Part E is attributable:

| Run name | adapter_active | top_k | chunk_size |
|---|---|---|---|
| `adapter-off-topk5` | False | 5 | 384 |
| `adapter-on-topk5` | True | 5 | 384 |
| `adapter-on-topk1` | True | 1 | 384 |

`adapter-off-topk5` versus `adapter-on-topk5` isolates the adapter.
`adapter-on-topk5` versus `adapter-on-topk1` isolates retrieval depth. If
you ever find yourself comparing `adapter-off-topk5` against
`adapter-on-topk1`, stop: two variables differ and the comparison
attributes nothing.

**Worked target output.** After the three calls, `RUNS` is a list of three
snapshot dicts and this prints:

```text
adapter-off-topk5  faithfulness=0.5   context_recall=1.0
adapter-on-topk5   faithfulness=1.0   context_recall=1.0
adapter-on-topk1   faithfulness=1.0   context_recall=0.8
```


In [ ]:
def tracked_eval(run_name: str, adapter_active: bool, top_k: int, chunk_size: int, extra_tags: list | None = None) -> dict:
    """Run one fully tracked Cordwell RAG evaluation.

    Composes everything from Parts B and C:
    1. wandb.init with project=PROJECT, name=run_name,
       config=make_config(adapter_active, top_k, chunk_size),
       tags=["rag-eval", "part-d"] plus any extra_tags.
    2. record_input_artifacts(run) so lineage is captured.
    3. results = ls.run_rag_eval(adapter_active=..., top_k=...,
       chunk_size=...)
    4. log_eval_metrics(run, results["aggregate"])
    5. Snapshot, finish, return:
       {"name": run_name, "id": run.id, "config": dict(run.config),
        "summary": dict(run.summary),
        "aggregate": results["aggregate"],
        "per_query": results["per_query"]}
    """
    raise NotImplementedError("Part D: implement tracked_eval")

In [ ]:
def run_experiment() -> list:
    """Run the three controlled evaluations from the Part D table.

    Call tracked_eval three times, in this order, with chunk_size=384
    for all three:
        "adapter-off-topk5": adapter_active=False, top_k=5
        "adapter-on-topk5":  adapter_active=True,  top_k=5
        "adapter-on-topk1":  adapter_active=True,  top_k=1
    Return the three snapshots as a list.
    """
    raise NotImplementedError("Part D: implement run_experiment")

In [ ]:
try:
    RUNS = run_experiment()
    for snap in RUNS:
        agg = snap["aggregate"]
        print(
            f"{snap['name']:<18} faithfulness={agg['faithfulness']:<5} "
            f"context_recall={agg['context_recall']}"
        )
except NotImplementedError as exc:
    RUNS = None
    if wandb.run is not None:
        wandb.run.finish()
    print(f"TODO: {exc}")


def _cfg_diff(a, b):
    keys = {"adapter_active", "top_k", "chunk_size"}
    return {k for k in keys if a["config"][k] != b["config"][k]}


check("D1: three runs collected", lambda: len(req(RUNS)) == 3)
check(
    "D2: run names match the experiment table",
    lambda: [s["name"] for s in req(RUNS)]
    == ["adapter-off-topk5", "adapter-on-topk5", "adapter-on-topk1"],
)
check(
    "D3: adapter pair differs by exactly one variable",
    lambda: _cfg_diff(req(RUNS)[0], RUNS[1]) == {"adapter_active"},
)
check(
    "D4: top_k pair differs by exactly one variable",
    lambda: _cfg_diff(req(RUNS)[1], RUNS[2]) == {"top_k"},
)
check(
    "D5: metrics match the locked deterministic values",
    lambda: (
        req(RUNS)[0]["aggregate"]["faithfulness"] == 0.5
        and RUNS[1]["aggregate"]["faithfulness"] == 1.0
        and RUNS[2]["aggregate"]["context_recall"] == 0.8
        and RUNS[1]["aggregate"]["context_recall"] == 1.0
    ),
)

## Part E: The comparison, and a recommendation with evidence

Objective 4, and the deliverable of the day. On the `local` backend you
would open the comparison view at http://localhost:8080 and select the
three runs. The same comparison built programmatically works on every
backend, and it is what a CI gate would consume in Week 7.

**Worked target output.**

```text
                run  adapter_active  top_k  faithfulness  context_recall  context_precision  answer_relevancy  abstention_rate
  adapter-off-topk5           False      5           0.5             1.0               0.24            0.7132           0.0000
   adapter-on-topk5            True      5           1.0             1.0               0.24            0.8346           0.1667
   adapter-on-topk1            True      1           1.0             0.8               0.80            0.8328           0.3333
```

Read it honestly, row against row, one variable at a time. Two traps are
planted. First, `abstention_rate` rose when the adapter came on; is that a
regression? Check `correct_abstention` in the aggregates before deciding.
Second, `context_precision` jumped to 0.80 at `top_k=1`, the best value in
the column, on the same row where recall fell. A single metric read in
isolation will happily tell you a worse system is better.


In [ ]:
def build_comparison(runs: list) -> pd.DataFrame:
    """Build the comparison table from the run snapshots.

    One row per run, in the given order, with columns exactly:
        run, adapter_active, top_k, chunk_size, faithfulness,
        context_recall, context_precision, answer_relevancy,
        abstention_rate
    "run" is the snapshot name; the three knobs come from the config;
    the five metrics come from the aggregate.
    """
    raise NotImplementedError("Part E: implement build_comparison")

In [ ]:
try:
    comparison_df = build_comparison(RUNS)
    print(comparison_df.to_string(index=False))
except (NotImplementedError, TypeError) as exc:
    comparison_df = None
    print(f"TODO: {exc}")

### Your recommendation

Fill in the `RECOMMENDATION` dict, then write three to five sentences in
the markdown cell after it. The dict is the machine-checkable core; the
prose is what you would actually send. A defensible recommendation names
the run to promote, the single baseline it was compared against, the one
variable that differed, the evidence metric with its delta, and a threshold
stated before you looked. Use 0.1 as the agreed threshold for this lab:
yesterday's judge noise analysis showed differences under 0.1 are not
findings.


In [ ]:
RECOMMENDATION = {
    # The run name to promote.
    "promote": None,
    # The single run it is compared against.
    "compared_to": None,
    # The one config key that differs between those two runs.
    "variable_changed": None,
    # The metric that carries the evidence, with its prefix.
    "evidence_metric": None,
    # promoted value minus baseline value, from the comparison table.
    "delta": None,
    # The evidence threshold agreed before looking at results.
    "threshold": 0.1,
    # Why adapter-on-topk1 is rejected: the one variable it isolates
    # and the metric it damaged.
    "rejected_run": None,
    "rejected_because": None,
}

**Your recommendation goes here.** Replace this text with three to
five sentences: the run you promote, the single run you compared it
against, the one variable that differed, the evidence metric and its
delta against the 0.1 threshold, and why the rejected run is rejected.


In [ ]:
check(
    "E1: comparison table has 3 rows and the required columns",
    lambda: req(comparison_df).shape[0] == 3
    and {
        "run", "adapter_active", "top_k", "chunk_size", "faithfulness",
        "context_recall", "context_precision", "answer_relevancy",
        "abstention_rate",
    }
    <= set(comparison_df.columns),
)
check(
    "E2: table values come from the runs",
    lambda: list(req(comparison_df)["faithfulness"]) == [0.5, 1.0, 1.0]
    and list(comparison_df["context_recall"]) == [1.0, 1.0, 0.8],
)
check(
    "E3: recommendation promotes on a one-variable comparison",
    lambda: req(RECOMMENDATION["promote"]) == "adapter-on-topk5"
    and RECOMMENDATION["compared_to"] == "adapter-off-topk5"
    and RECOMMENDATION["variable_changed"] == "adapter_active",
)
check(
    "E4: evidence clears the pre-stated threshold",
    lambda: req(RECOMMENDATION["evidence_metric"]) == "generation/faithfulness"
    and abs(req(RECOMMENDATION["delta"]) - 0.5) < 1e-9
    and RECOMMENDATION["delta"] > RECOMMENDATION["threshold"]
    and RECOMMENDATION["rejected_run"] == "adapter-on-topk1",
)

## Part F: The per-query table and the worst failures

Aggregate metrics tell you something is wrong. The table tells you which
queries. You will log a `wandb.Table` of the baseline run
(`adapter-off-topk5`, the run with real failures in it) and then find the
three worst rows, which is exactly what sorting by a metric ascending does
in the UI.

**Worked target output.**

```text
Worst three queries: ['q12', 'q11', 'q06']
q12  rel=0.05    What is the horsepower of the Cordwell RidgeRunner riding mower?
q11  rel=0.1286  Does Cordwell price match online retailers on power tools?
q06  rel=0.79    How much weight can the large black GripFast drywall anchor hold?
```

Note what the table surfaces that the aggregate hid: the two worst rows
are the unanswerable questions. The base model answered them anyway, with
text assembled from whatever it retrieved. The aggregate said
`faithfulness 0.5`; the table says where the bodies are buried.

Pass `columns` explicitly and use `add_data` row by row. A list of dicts
is not a valid data argument; that is a common error in older examples.


In [ ]:
TABLE_COLUMNS = [
    "query_id", "question", "answer", "sources",
    "faithfulness", "answer_relevancy", "abstained",
]


def build_per_query_table(per_query: list) -> wandb.Table:
    """Build a wandb.Table of per-query results.

    Create wandb.Table(columns=TABLE_COLUMNS), then add one row per
    result with add_data, in this order: query_id, question, answer,
    sources joined with ", ", faithfulness, answer_relevancy,
    abstained.
    """
    raise NotImplementedError("Part F: implement build_per_query_table")


def find_worst_queries(per_query: list, n: int = 3) -> list:
    """Return the query_ids of the n worst rows by answer relevancy.

    Sort ascending by (answer_relevancy, query_id); the query_id
    tiebreak keeps the result deterministic. This baseline run has no
    abstentions, so every relevancy is a number.
    """
    raise NotImplementedError("Part F: implement find_worst_queries")

In [ ]:
try:
    baseline_per_query = RUNS[0]["per_query"]
    per_query_table = build_per_query_table(baseline_per_query)

    _run = wandb.init(project=PROJECT, name="failure-analysis-baseline", tags=["failure-analysis"])
    _run.log({"eval/per_query": per_query_table})
    _run.finish()

    WORST_THREE = find_worst_queries(baseline_per_query)
    print(f"Worst three queries: {WORST_THREE}")
    by_id = {r["query_id"]: r for r in baseline_per_query}
    for qid in WORST_THREE:
        r = by_id[qid]
        print(f"{qid}  rel={r['answer_relevancy']:<7} {r['question']}")
except (NotImplementedError, TypeError) as exc:
    per_query_table = None
    WORST_THREE = None
    if wandb.run is not None:
        wandb.run.finish()
    print(f"TODO: {exc}")

check(
    "F1: table has the exact columns and 12 rows",
    lambda: req(per_query_table).columns == TABLE_COLUMNS
    and len(per_query_table.data) == 12,
)
check(
    "F2: worst three identified",
    lambda: req(WORST_THREE) == ["q12", "q11", "q06"],
)
check(
    "F3: the two worst rows are the unanswerable questions",
    lambda: all(
        not next(r for r in RUNS[0]["per_query"] if r["query_id"] == qid)["answerable"]
        for qid in req(WORST_THREE)[:2]
    ),
)

## Part G (stretch): A small grid sweep

Objective 5. A sweep is a controlled experiment executed automatically:
every combination becomes one recorded run. On a backend with a server the
native API is:

```python
sweep_config = {
    "method": "grid",
    "metric": {"name": "retrieval/context_recall", "goal": "maximize"},
    "parameters": {
        "top_k": {"values": [1, 3, 5]},
        "chunk_size": {"values": [256, 384, 512]},
    },
}
sweep_id = wandb.sweep(sweep_config, project=PROJECT)
wandb.agent(sweep_id, function=one_sweep_run, count=9)
```

`wandb.sweep` registers the sweep on the server and `wandb.agent` pulls
configurations from it, so neither works offline (they raise `UsageError`).
The manual grid below is the same experiment expressed as a loop, runs on
every backend, and makes the mechanics visible: nine configurations, nine
tracked runs, one collected table. If a continuous parameter like
temperature or learning rate ever enters a sweep config, the distribution
is `log_uniform_values` with literal min and max; `log_uniform` samples
between `exp(min)` and `exp(max)` and fails silently.

**Worked target output.** Nine rows; recall ranges from 0.6 at the
smallest chunking with `top_k=1` up to 1.0 for five of the nine
combinations, and the frontier pick is:

```text
Best config: {'chunk_size': 256, 'top_k': 3}
```


In [ ]:
def run_manual_sweep() -> pd.DataFrame:
    """Run the 3x3 grid as nine tracked runs and collect the results.

    For every chunk_size in [256, 384, 512] and top_k in [1, 3, 5]
    (that nesting order), call tracked_eval with adapter_active=True,
    run_name f"sweep-cs{chunk_size}-k{top_k}", and
    extra_tags=["sweep"]. Collect one row per run with columns:
        run, chunk_size, top_k, context_recall, context_precision,
        abstention_rate
    Return the rows as a DataFrame.
    """
    raise NotImplementedError("Part G (stretch): implement run_manual_sweep")

In [ ]:
def pick_best_config(sweep_df: pd.DataFrame) -> dict:
    """Pick one configuration from the sweep frontier, deterministically.

    Rule, stated before looking, in priority order:
    1. Highest context_recall (the metric the sweep maximizes).
    2. Among ties, the smallest top_k: less context per query is
       cheaper at inference time for the same recall.
    3. Among remaining ties, the highest context_precision.
    Return {"chunk_size": ..., "top_k": ...} as plain ints.
    """
    raise NotImplementedError("Part G (stretch): implement pick_best_config")

In [ ]:
try:
    sweep_df = run_manual_sweep()
    print(sweep_df.to_string(index=False))
    BEST_SWEEP = pick_best_config(sweep_df)
    print(f"Best config: {BEST_SWEEP}")
except (NotImplementedError, TypeError) as exc:
    sweep_df = None
    BEST_SWEEP = None
    if wandb.run is not None:
        wandb.run.finish()
    print(f"TODO: {exc}")

check("G1: sweep produced nine runs", lambda: req(sweep_df).shape[0] == 9)
check(
    "G2: recall varies across the grid",
    lambda: req(sweep_df)["context_recall"].max() == 1.0
    and sweep_df["context_recall"].min() == 0.6,
)
check(
    "G3: frontier pick follows the stated rule",
    lambda: req(BEST_SWEEP) == {"chunk_size": 256, "top_k": 3},
)

**What the sweep can and cannot tell you.** It found that
`chunk_size=256, top_k=3` matches the recall of the Part D configuration
with fewer retrieved chunks per query and better precision. What it cannot
tell you is whether that config should ship: the sweep held the adapter,
the eval set, and the judge constant, and it optimized one retrieval
metric. Before promoting the sweep winner you would re-run the Part D
comparison at the new configuration and check the generation metrics
too. A sweep is a hypothesis generator with excellent record keeping, not
a decision.


In [ ]:
report()

## What you built

Every evaluation this week now leaves a record: a config that passes the
re-run test, metrics named by stage, versioned artifacts with lineage, and
a comparison that attributes each metric change to the one variable that
caused it. The Part E writeup format, a claim plus a comparison plus the
evidence with run links, is exactly the format the capstone presentation
needs.

If you ran the `offline` backend, everything you logged is in
`./wandb/offline-run-*` and `wandb sync <dir>` pushes any of it to a
server later. If you ran `local`, open http://localhost:8080, select the
three Part D runs, and confirm the comparison view shows what your
DataFrame shows. Same data, same conclusion; the UI is just faster to
read.

Monday covers MLflow and the tracking landscape. The concepts you used
today, config completeness, artifact lineage, one-variable comparison,
transfer completely. The API does not, and that is fine; you now know
which parts are the tool and which parts are the discipline.
